In [1]:
import pandas as pd
import torch
import torch.nn as nn
from model import GPT

In [2]:
data = pd.read_csv("C://Users//USER//Downloads//archive (6)//train.csv")
data= data.dropna()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
text = "\n".join(data["text"].astype(str))
chars = sorted(list(set(text)))

char2idx = {char:idx for idx,char in enumerate(chars)}
idx2char = {idx:char for idx,char in enumerate(chars)}


In [4]:
encoded = torch.tensor([char2idx[c] for c in text], dtype=torch.long)

In [5]:
n= int(0.9*len(encoded))
train = encoded[:n]
test = encoded[n:]

In [6]:
def get_batch(data, context_len, batch_size):
    
    ix = torch.randint(len(data)-context_len-1,(batch_size,))
    X = torch.stack([data[i:i+context_len] for i in ix])
    Y = torch.stack([data[i+1:i+1+context_len] for i in ix])
    return X,Y

In [7]:
vocab_size = 65
batch_size = 32
dropout = 0.1

model_data = [
    {"name":"Tiny","d_model": 64,  "num_heads": 2, "num_layer": 2, "context_len":128},
    {"name": "Small","d_model": 128, "num_heads": 4, "num_layer": 4, "context_len":256},
    {"name":"Medium","d_model": 256, "num_heads": 8, "num_layer": 6, "context_len":256}]

def count_non_embedding_params(model):
    return sum(p.numel() for name, p in model.named_parameters() if 'embed' not in name and 'lm_head' not in name)

In [8]:

criterion = nn.CrossEntropyLoss()
epochs =3000

    
results =[]
for model_par in model_data:
    model = GPT(vocab_size,model_par["d_model"],model_par["context_len"],model_par["num_heads"],model_par["num_layer"],dropout).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr =3e-4)
    best_val_scores= float("inf")
    for epoch in range(epochs):
        model.train()
        X_train,Y_train= get_batch(train,model_par["context_len"],batch_size)
        x_train = X_train.to(device) 
        y_train = Y_train.to(device)
        logits = model(x_train)
        
        B,T,C = logits.shape
        loss = criterion(logits.reshape(B*T,C),y_train.reshape(B*T))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch % 500 == 0:
            
            model.eval()
            with torch.no_grad():
                X_test,Y_test = get_batch(test,model_par["context_len"],batch_size)
                X_test = X_test.to(device)
                Y_test = Y_test.to(device)
                
                logits = model(X_test)
                B,T,C = logits.shape
                val_loss = criterion(logits.reshape(B*T,C),Y_test.reshape(B*T))
                
                if val_loss.item()<best_val_scores:
                    best_val_loss = val_loss.item()
                   
      
    results.append({
        "Experiment": "Parameter Scaling",
        "Model": model_par["name"],
        "d_model": model_par["d_model"],
        "Layers": model_par["num_layer"],
        "Heads": model_par["num_heads"],
        "Parameters": count_non_embedding_params(model),
        "Validation Loss": best_val_loss})
        

pd.DataFrame(results).to_csv("scaling_results.csv", index=False)          

In [ ]:
data_percentages = [0.10, 0.25, 0.50, 1.00]
steps = 3000
seq_len = 256
d_model = 128
for p in data_percentages:
    print(f"Training Small Model on {int(p*100)}% Data")
    subset_train = train[:int(len(train)*p)]
    model = GPT(vocab_size,d_model,seq_len,4,4,dropout).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

    best_val_loss = float("inf")

    for step in range(steps):

        model.train()
        X_train, Y_train = get_batch(subset_train,seq_len,batch_size)
        X_train = X_train.to(device)
        Y_train = Y_train.to(device)

        logits = model(X_train)

        B, T, C = logits.shape

        loss = criterion(logits.reshape(B*T, C),Y_train.reshape(B*T))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 500 == 0:
            model.eval()
            with torch.no_grad():

                X_test, Y_test = get_batch(test,seq_len,batch_size)
                X_test = X_test.to(device)
                Y_test = Y_test.to(device)

                logits = model(X_test)
                B, T, C = logits.shape

                val_loss = criterion(logits.reshape(B*T, C),Y_test.reshape(B*T))

                if val_loss.item() < best_val_loss:
                    best_val_loss = val_loss.item()

    results.append({
        "Experiment": "Data Scaling",
        "Dataset_size": int(len(subset_train)),
        "Model": "Small",
        "d_model": 128,
        "Layers": 4,
        "Heads": 4,
        "Parameters": count_non_embedding_params(model),
        "Validation Loss1 ": best_val_loss})
    
pd.DataFrame(results).to_csv("scaling_results.csv", index=False)      
  

Training Small Model on 10% Data
Training Small Model on 25% Data
Training Small Model on 50% Data
Training Small Model on 100% Data
